# KAT-VAD v2 — **Amendment 20: can `B` drop CRN?** A2-ign vs A3-ign on T2 (non-inferiority)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§31 (Amendment 20, Z1–Z10)** — it must be in the upload and be
**committed before this notebook trains anything**. Plan: `.project/plans/katvad-v2-crn-drop.md`.
A2-ign = A3-ign (E4-0's recipe) with `v2.crn = none` — **nothing else changes** (proven in step 1 before any GPU time).
Inputs = E3's **A2** caches as baked (`A2_S`, T2 + DoTA-CAP): no extraction, no bake. The control is **E4-0's five
A3-ign runs**, read from `Thesis-V2/outputs/v2_e4_0/`, never retrained.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps; refuses an upload without §31 / `crn_drop_readout` | — |
| 1 | preflight: E4-0 runs + reads, E3 A0 diags + A2 caches, config diff = exactly `{v2.crn}` (Z1, Z10(b)(c)) | no |
| 2 | stage the v2 dir, KNN, the `A2_S` inputs, DoTA s1 + labels to VM disk | no |
| 3 | **train A2-ign × 5** (seeds 2024–2028), chunked + resumable, J10 | **yes** |
| 4 | **diagnostics** (O1′ vs E3's A0 of the same seed) | yes |
| 5 | **protocol B** for A2-ign on `dota_cap_dev` | yes |
| 6 | **`crn_drop_readout`** — the §31 Z7 rule → stop | no |

Nothing reads T2-test, DoTA-eval or DoTA-CAP-eval. Every step resumes: re-run the cell after a disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_CRNDROP_ARM'), 'this upload predates Amendment 20 -- re-upload branch v2'
assert (REPO / 'core/tools/crn_drop_readout.py').is_file(), 'upload has no core/tools/crn_drop_readout.py'
ADDENDUM = REPO / 'core/docs/v2/PREREG_ADDENDUM.md'
assert '## 31. Amendment 20' in ADDENDUM.read_text(encoding='utf-8'), 'upload has no addendum §31 (drop-CRN rule)'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # E3 / E4-0's recipe
SEEDS      = list(constants.V2_E3_SEEDS)                 # 2024-2028, paired with E4-0's A3-ign
CONTROL    = constants.V2_CRNDROP_CONTROL                # 'A3ign'
ARM        = constants.V2_CRNDROP_ARM                    # 'A2ign'
ANCHOR     = constants.V2_E4_0_ANCHOR_MODE               # 'ignore' (as A3-ign)
ENC        = constants.VIDEOMAE_ENCODER_S                # V2-S (as A3-ign)
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]             # 384
CAP_SPLIT  = constants.V2_SPLIT_DOTA_CAP_DEV
TAG        = constants.V2_CRNDROP_CACHE_TAG              # 'A2_S'

DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json (class names)
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
ALIGN     = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap' / 'dota_cap_alignment.json'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE  = Path(DRIVE_V2) / 'cache'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
A3_T2     = V2_CACHE / 'v2' / 'A3_R2_S' / DATASET         # A3 / A3-ign's input (train-split reference)
A2_T2     = V2_CACHE / 'v2' / TAG / DATASET               # A2-ign's T2 input = E3's A2 bake (Z1)
A2_CAP    = V2_CACHE / 'v2' / TAG / 'DoTA_CAP_s3_from_s1' # A2-ign's DoTA-CAP input = E3's A2 bake (Z1)
E3        = constants.OUTPUT_ROOT / 'v2_e3'               # A0 diags (O1'), A2 pb (printed), p_T2
E40       = constants.OUTPUT_ROOT / 'v2_e4_0'             # control: A3ign/s<seed>, its pb read + diags
OUT       = constants.OUTPUT_ROOT / 'v2_crndrop'          # this step: A2ign/, REPORTS/
LOCAL     = Path('/content/crndrop')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('V2_DATA', 'KNN_V2', 'A3_T2', 'A2_T2', 'A2_CAP', 'E3', 'E40', 'OUT'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 3-5 need one'
df -h /content | tail -1

In [ ]:
import json
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    # Copy a Drive dir (only ``ids`` + its manifests when given) / file to VM disk once, bit-identical (C2).
    t0 = time.time()
    if src.is_dir() and ids is None:
        if not dst.exists():
            shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    elif not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    # Copy a VM-local result to Drive ATOMICALLY (tmp + fsync + rename; lesson (bi)).
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    # The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE checkpoint {path} ({path.stat().st_size / 2**20:.0f} MiB): {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def verified(d):
    # J10: finished, and the checkpoint's own step equals the metrics' last step.
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last and last['epoch'] + 1 == NUM_EPOCHS
                and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'])


def config_diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out

## 1. Preflight — the control, the config diff (Z1, Z10(b)) and the `A2_S` caches (Z10(c)), **before** any GPU time

In [ ]:
import tempfile

from core.config import load_config
from core.data.v2_splits import load_split
from core.tools import crn_drop_readout, e3_readout, position_prior, protocol_b_eval, v2_dataset, v2_diagnostics

EXPECTED_DIFF = {'v2.crn'}
cap_dev = load_split(CAP_SPLIT)                                          # sha1-checked
assert len(cap_dev) == 569, len(cap_dev)
for path in (V2_DATA / v2_dataset.MANIFEST_FILENAME, KNN_V2, A3_T2 / constants.V2_INPUT_MANIFEST_FILENAME,
             A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME, A2_CAP / constants.V2_INPUT_MANIFEST_FILENAME,
             LABELS / constants.DEFS_FILENAME, ALIGN,
             e3_readout.pb_dir(E40, CONTROL, CAP_SPLIT) / protocol_b_eval.CLIP_SCORES_NPZ,
             E3 / e3_readout.REPORTS / e3_readout.PRIOR_DIR / CAP_SPLIT / position_prior.PRIOR_NPZ):
    assert path.exists(), f'missing {path}'
assert all((A2_CAP / f'{v}.npy').exists() for v in cap_dev), f'{A2_CAP} does not cover dota_cap_dev'

# Z10(c): E3's A2 bake = no CRN, V2-S, on A3's train split
a3 = json.loads((A3_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
for d in (A2_T2, A2_CAP):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    print(f"{d.name}: crn {m['crn']} motion {m['motion']} width {m['width']}")
    assert m['crn'] == constants.V2_OFF and m['motion'] == ENC and m['width'] == 512 + MOTION_DIM, m
    assert m['train_ids_sha1'] == a3['train_ids_sha1'], f'{d} fitted on another train split than A3 -- STOP'

E40_CFG = {}
for seed in SEEDS:
    d = E40 / CONTROL / f's{seed}'
    assert verified(d), f'E4-0 {CONTROL} s{seed} is not a finished run on Drive (J10)'
    assert (d / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists(), f'E4-0 {CONTROL} s{seed} has no diag'
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    assert a0.exists(), f'missing E3 A0 diag {a0} (O1′ reference)'
    E40_CFG[seed] = yaml.safe_load((d / 'config.yaml').read_text())
    assert E40_CFG[seed]['loss']['dvs_anchor_mode'] == ANCHOR, f'E4-0 {CONTROL} s{seed} is not {ANCHOR}'

SETS = ['train.stage=2', f'train.num_epochs={NUM_EPOCHS}', 'train.amp=true', f'data.dataset={DATASET}',
        f'model.score_head_kernel={KERNEL}', f'loss.mil_topk_pct={TOPK_PCT}', 'kip.enabled=false',
        f'v2.crn={constants.V2_OFF}', f'v2.motion={ENC}', f'model.motion_dim={MOTION_DIM}',
        f'loss.dvs_anchor_mode={ANCHOR}']
for seed in SEEDS:
    with tempfile.TemporaryDirectory() as tmp:     # the config A2-ign WILL train with, built now
        cfg = load_config(None, [*SETS, f'train.seed={seed}'])
        cfg.save_yaml(Path(tmp) / 'config.yaml')
        planned = yaml.safe_load((Path(tmp) / 'config.yaml').read_text())
    diff = config_diff(E40_CFG[seed], planned)
    assert diff == EXPECTED_DIFF, f's{seed}: A2-ign would differ from E4-0 A3-ign in {diff}'
print(f'every seed: A2-ign = E4-0 A3-ign + {sorted(EXPECTED_DIFF)} (R2 -> none), nothing else | preflight OK')

## 2. Stage inputs to VM disk (bit-identical copies, C2)

In [ ]:
L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_KNN = stage(KNN_V2, LOCAL / 'stage' / 'knn_v2.npz')
L_T2 = stage(A2_T2, LOCAL / 'stage' / 'A2_T2')
L_CAP = stage(A2_CAP, LOCAL / 'stage' / 'A2_CAP', cap_dev)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap', cap_dev)
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
print('staged')

## 3. Train A2-ign × 5 — chunked, resumable, verified (Z1, Z10(a)(b))

E4-0's recipe without CRN. After every chunk the written `config.yaml` is diffed against E4-0's A3-ign of the same
seed (must be exactly `{v2.crn}`). Skip/resume reads the checkpoint's own `global_step` (J10).
**Re-run this cell after a disconnect.**

In [ ]:
def run_dir(seed):
    return OUT / ARM / f's{seed}'


def train_cmd(seed, out_dir):
    return [sys.executable, '-m', 'core.train', *[x for s in [*SETS, f'train.seed={seed}'] for x in ('--set', s)],
            '--data-dir', L_V2, '--clip-dir', L_T2, '--knn-cache', L_KNN, '--output-dir', out_dir]


def sync(local, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive_dir / name)


def check_diff(seed, cfg_path):
    diff = config_diff(E40_CFG[seed], yaml.safe_load(cfg_path.read_text()))
    assert diff == EXPECTED_DIFF, f'A2-ign s{seed} differs from E4-0 A3-ign in {diff} -- STOP'


durations = []
todo = [s for s in SEEDS if not verified(run_dir(s))]
print(f'{len(todo)} runs to train in this session')
for k, seed in enumerate(todo):
    drive_dir, local = run_dir(seed), LOCAL / 'runs' / ARM / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    if not (local / 'checkpoint_last.pt').exists() and ckpt_step(drive_dir / 'checkpoint_last.pt') == -1:
        stamp = time.strftime('%Y%m%d-%H%M%S')          # truncated by an interrupted copy: keep, never delete
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            if (drive_dir / name).exists():
                os.replace(drive_dir / name, drive_dir / f'{name}.corrupt-{stamp}')
        print(f'{ARM} s{seed}: Drive checkpoint unreadable -> kept as *.corrupt-{stamp}; retraining (same seed)')
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    t0 = time.time()
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(seed, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {ARM} s{seed}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{ARM}_s{seed}_train.log', quiet_steps=True)
        check_diff(seed, local / 'config.yaml')
        sync(local, drive_dir)
    if not verified(drive_dir):                          # never re-copy a verified checkpoint
        sync(local, drive_dir)
    assert verified(drive_dir), f'{ARM} s{seed}: J10 FAILS on Drive'
    check_diff(seed, drive_dir / 'config.yaml')
    durations.append(time.time() - t0)
    eta = sum(durations) / len(durations) * (len(todo) - k - 1)
    print(f'{ARM} s{seed}: step {metrics_last(drive_dir / "metrics.jsonl")["global_step"]}, '
          f'{durations[-1] / 60:.1f} min; ETA this session {eta / 3600:.1f} h')
missing = [s for s in SEEDS if not verified(run_dir(s))]
print('all 5 A2-ign runs verified' if not missing else f'still to train: {missing}')

## 4. Diagnostics — A2-ign (O1′ vs E3's A0 of the same seed; Z6)

A3-ign's diags are E4-0's (same code, `t2_val_reads`), never recomputed.

In [ ]:
def diagnose(seed):
    out = OUT / ARM / f's{seed}' / e3_readout.DIAG_DIR
    if (out / v2_diagnostics.DIAG_JSON).exists():
        return
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', f'{ARM}_s{seed}',
         run_dir(seed) / 'checkpoint_last.pt',
         '--data-dir', L_V2, '--t2-input-dir', L_T2, '--dota-input-dir', L_CAP,
         '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS, '--dota-split', CAP_SPLIT,
         '--a0-diag', a0, '--out-dir', out, '--device', 'auto'], log=LOCAL / f'{ARM}_s{seed}_diag.log')


assert not [s for s in SEEDS if not verified(run_dir(s))], 'train every A2-ign run first'
for seed in SEEDS:
    diagnose(seed)
print('diagnosed:', sum((OUT / ARM / f's{s}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists()
                        for s in SEEDS), 'of 5')

## 5. Protocol B for A2-ign on `dota_cap_dev` (A3-ign's is E4-0's)

In [ ]:
out = e3_readout.pb_dir(OUT, ARM, CAP_SPLIT)
if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
    print('reusing', out)
else:
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', L_CAP, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', CAP_SPLIT, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{ARM}_{CAP_SPLIT}_pb.log')
print((out / protocol_b_eval.READOUT_MD).read_text())

## 6. Read-out and the drop-CRN rule (§31 Z7) — no GPU

In [ ]:
run([sys.executable, '-m', 'core.tools.crn_drop_readout', '--e3-dir', E3, '--e4-dir', E40, '--out-dir', OUT,
     '--alignment', ALIGN])
READ = json.loads((OUT / e3_readout.REPORTS / crn_drop_readout.READOUT_JSON).read_text())
print((OUT / e3_readout.REPORTS / crn_drop_readout.READOUT_MD).read_text())
print('CRN DROPPED: B = A2-ign' if READ['decision']['drop_crn']
      else f"A3-ign stays (failed: {READ['decision']['failed']})")

## 7. Record — then **stop**

Paste `Thesis-V2/outputs/v2_crndrop/REPORTS/crn_drop_readout.md` back to Claude. The decision is mechanical (§31 Z7):
nothing is re-run, re-seeded or re-tuned after reading it. Whatever it says, no sealed set is read here (Z9).